<a href="https://colab.research.google.com/github/HC-AIGroup/CS5330_F26/blob/main/module3_part3/Otsu_vs_Triangle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Otsu vs. Triangle: which automatic threshold works when?
Both methods choose the threshold **T** automatically. We test them on two images:
1. **Coins**: object and background are both large → two peaks in the histogram
2. **Night sky**: background is huge, stars are tiny and rare → one peak + long tail

The images are generated in code, so no download is needed.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## Step 1: Make the two test images
We also keep the "true" object mask so we can check which method is more accurate.

In [ ]:
def make_coins():
    img = np.full((300, 400), 70, np.float32)              # gray background
    truth = np.zeros((300, 400), np.uint8)
    for (x, y, r) in [(80, 80, 45), (200, 90, 50), (320, 80, 40),
                      (100, 210, 50), (230, 220, 45), (340, 210, 42)]:
        cv2.circle(truth, (x, y), r, 1, -1)
    img[truth == 1] = 170                                   # bright coins
    img += rng.normal(0, 18, img.shape)                     # camera noise
    img = cv2.GaussianBlur(img, (5, 5), 0)                  # soft edges
    return np.clip(img, 0, 255).astype(np.uint8), truth

def make_night():
    img = np.full((300, 400), 40, np.float32)               # dark night sky
    truth = np.zeros((300, 400), np.uint8)
    pts = rng.integers([20, 20], [380, 280], size=(20, 2))
    for (x, y) in pts:                                      # 20 tiny stars
        star = np.zeros_like(truth)
        cv2.circle(star, (int(x), int(y)), 4, 1, -1)
        img[star == 1] = rng.uniform(80, 230)               # each star has a different brightness
        truth |= star
    img = cv2.GaussianBlur(img, (0, 0), 2)                  # soft edges
    img += rng.normal(0, 10, img.shape)                     # camera noise
    return np.clip(img, 0, 255).astype(np.uint8), truth

coins, coins_truth = make_coins()
night, night_truth = make_night()
print(f"Coins: object covers {coins_truth.mean()*100:.1f}% of the image")
print(f"Night: stars cover {night_truth.mean()*100:.1f}% of the image")

## Step 2: Threshold each image with Otsu and Triangle
`0` is ignored: the method computes its own T and returns it.

In [ ]:
def run_both(img):
    T_otsu, bin_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    T_tri,  bin_tri  = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
    return T_otsu, bin_otsu, T_tri, bin_tri

def accuracy(binary, truth):
    """Percent of pixels labelled correctly (object vs. background)."""
    return ((binary > 0) == (truth == 1)).mean() * 100

def false_object(binary, truth):
    """How many background pixels were wrongly called object."""
    return int(((binary > 0) & (truth == 0)).sum())

results = {}
for name, img, truth in [("Coins", coins, coins_truth), ("Night", night, night_truth)]:
    T_o, b_o, T_t, b_t = run_both(img)
    results[name] = (img, T_o, b_o, T_t, b_t)
    print(f"\n{name} image")
    print(f"  Otsu     T = {T_o:5.0f}   accuracy = {accuracy(b_o, truth):5.1f}%   "
          f"background wrongly called object = {false_object(b_o, truth):6d} px")
    print(f"  Triangle T = {T_t:5.0f}   accuracy = {accuracy(b_t, truth):5.1f}%   "
          f"background wrongly called object = {false_object(b_t, truth):6d} px")

## Step 3: Look at the histograms and the results

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, name in enumerate(["Coins", "Night"]):
    img, T_o, b_o, T_t, b_t = results[name]
    ax = axes[row]
    ax[0].imshow(img, cmap="gray", vmin=0, vmax=255); ax[0].set_title(f"{name}: original")
    ax[1].hist(img.ravel(), bins=256, range=(0, 255), color="gray")
    ax[1].axvline(T_o, color="tab:blue", lw=2.5, label=f"Otsu T={T_o:.0f}")
    ax[1].axvline(T_t, color="tab:red",  lw=2.5, ls="--", label=f"Triangle T={T_t:.0f}")
    ax[1].set_yscale("log"); ax[1].legend(); ax[1].set_title("Histogram (log scale)")
    ax[1].set_xlabel("Gray level"); ax[1].set_ylabel("Number of pixels")
    ax[2].imshow(b_o, cmap="gray"); ax[2].set_title(f"Otsu (T={T_o:.0f})", color="tab:blue")
    ax[3].imshow(b_t, cmap="gray"); ax[3].set_title(f"Triangle (T={T_t:.0f})", color="tab:red")
    for a in (ax[0], ax[2], ax[3]): a.axis("off")
plt.tight_layout()
plt.show()

## Discussion questions
1. On the **coins** image, which method gives the cleaner result? Why does Otsu work well here?
2. On the **night** image, where does Otsu put T? What happens to the dark sky?
3. Look at the histograms: which image has two peaks, and which has one peak and a long tail?
4. **Try it:** change the coin brightness from 170 to 110. Which method breaks first?